#### **04 — Seleção de Variáveis**

**Objetivo:** reduzir o conjunto de variáveis usando IV, correlação de Spearman e VIF, sempre com dados de treino.

**Entrada:** `data/02_gold/*_woe.parquet` e `binning_summary.csv`  
**Saída:** `variaveis_selecionadas.json` e `data/02_gold/*_selecionado.parquet`

Critérios:
- IV mínimo: `0.02`
- Correlação máxima entre variáveis: `0.70`
- VIF máximo: `5`

In [1]:
import json
import pandas as pd
import numpy as np

from modelo_credito.selection import (
    calculate_vif,
    filter_by_iv,
    remove_high_correlation,
    select_by_vif,
)

pd.set_option("display.max_columns", None)


##### **1. Carga dos dados WoE**

In [2]:
treino = pd.read_parquet("../data/02_gold/treino_woe.parquet")
validacao = pd.read_parquet("../data/02_gold/validacao_woe.parquet")
teste = pd.read_parquet("../data/02_gold/teste_woe.parquet")

binning_summary = pd.read_csv("../data/02_gold/binning_summary.csv")

metadata = ["ID", "SAFRA"]
target = "default"

features = [col for col in treino.columns if col not in metadata + [target]]
iv = binning_summary.set_index("name")["iv"]

print("Variáveis disponíveis:", len(features))

Variáveis disponíveis: 35


#### **2. Filtro por Information Value**

In [3]:
iv_minimo = 0.02
selecionadas = filter_by_iv(features, iv, threshold=iv_minimo)

print("Após filtro de IV:", len(selecionadas))
print(selecionadas)


Após filtro de IV: 26
['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'SUM_BILL', 'AVG_BILL', 'MAX_BILL', 'SUM_PAY', 'AVG_PAY', 'MAX_PAY', 'UTILIZACAO', 'UTIL_MEDIA', 'PCT_PAGAMENTO', 'FREQ_ATRASO', 'MAX_ATRASO', 'LIMIT_PER_CAPITA']


#### **3. Remoção de redundância por Spearman**

Pares com `|correlação| > 0.70`: remove a variável de menor IV.

In [4]:
limite_correlacao = 0.70
selecionadas = remove_high_correlation(
    treino,
    selecionadas,
    iv,
    threshold=limite_correlacao,
)

print("Após filtro de correlação:", len(selecionadas))
print(selecionadas)


Removida AVG_PAY por correlação 1.000 com outra variável
Removida AVG_BILL por correlação 1.000 com outra variável
Removida LIMIT_BAL por correlação 0.916 com outra variável
Removida MAX_ATRASO por correlação 0.885 com outra variável
Removida MAX_PAY por correlação 0.811 com outra variável
Removida UTILIZACAO por correlação 0.771 com outra variável
Removida PAY_AMT4 por correlação 0.705 com outra variável
Removida PAY_AMT5 por correlação 0.700 com outra variável
Após filtro de correlação: 18
['AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT6', 'SUM_BILL', 'MAX_BILL', 'SUM_PAY', 'UTIL_MEDIA', 'PCT_PAGAMENTO', 'FREQ_ATRASO', 'LIMIT_PER_CAPITA']


#### **4. VIF**

O VIF é calculado nas colunas WoE do treino. Remove-se iterativamente a variável de maior VIF até todas ficarem abaixo do limite.

In [5]:
limite_vif = 5.0
selecionadas = select_by_vif(treino, selecionadas, threshold=limite_vif)

print("Após filtro de VIF:", len(selecionadas))
print(selecionadas)


Removida FREQ_ATRASO por VIF 7.95
Removida SUM_PAY por VIF 5.20
Após filtro de VIF: 16
['AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT6', 'SUM_BILL', 'MAX_BILL', 'UTIL_MEDIA', 'PCT_PAGAMENTO', 'LIMIT_PER_CAPITA']


#### **5. Tabela final de seleção**

In [6]:
vif_final = calculate_vif(treino, selecionadas).sort_values(ascending=False)

tabela_final = pd.DataFrame({
    "variavel": selecionadas,
    "iv": [iv.get(col, np.nan) for col in selecionadas],
    "vif": [vif_final[col] for col in selecionadas],
}).sort_values("iv", ascending=False)

tabela_final


,variavel,iv,vif
1,PAY_0,0.833657,1.929373
2,PAY_2,0.503322,3.046290
3,PAY_3,0.376393,2.759573
4,PAY_4,0.343313,2.922619
5,PAY_5,0.334614,2.572742
6,PAY_6,0.263786,1.823002
7,PAY_AMT1,0.175904,2.978403
15,LIMIT_PER_CAPITA,0.147221,1.454246
8,PAY_AMT2,0.144612,2.071042
13,UTIL_MEDIA,0.117589,1.347942


#### **6. Gravação das bases selecionadas**

In [7]:
with open("../data/02_gold/variaveis_selecionadas.json", "w") as file:
    json.dump(selecionadas, file, indent=2)

treino[metadata + [target] + selecionadas].to_parquet("../data/02_gold/treino_selecionado.parquet")
validacao[metadata + [target] + selecionadas].to_parquet("../data/02_gold/validacao_selecionado.parquet")
teste[metadata + [target] + selecionadas].to_parquet("../data/02_gold/teste_selecionado.parquet")

print("Variáveis selecionadas:", len(selecionadas))
print("Arquivos gravados em data/02_gold")

Variáveis selecionadas: 16
Arquivos gravados em data/02_gold
